# 部署準備（ML 工程師）

照 DS 的 `model_report.json` 重建模型，量單筆推論延遲。

In [ ]:
import pandas as pd
import numpy as np
import json, time
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

Path("output").mkdir(exist_ok=True)
df = pd.read_csv("input/cleaned.csv")
spec = json.loads(Path("input/model_report.json").read_text(encoding="utf-8"))
FEATURES = spec["features"]
print("照 DS 的報告重建模型：", spec["model_type"], FEATURES, "val AUC", spec["val_auc"])

In [ ]:
X, y = df[FEATURES], df["default"]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train)

# 單筆推論延遲：驗證集逐筆各跑一次，取 p50／p95
lat = []
for i in range(len(X_val)):
    row = X_val.iloc[[i]]
    t = time.perf_counter()
    model.predict_proba(row)
    lat.append((time.perf_counter() - t) * 1000)
p50, p95 = np.percentile(lat, 50), np.percentile(lat, 95)
print(f"n={len(lat)}  p50={p50:.3f}ms  p95={p95:.3f}ms")

In [ ]:
THRESHOLD_MS = 200
report = {
    "val_auc": spec["val_auc"],
    "inference_ms_p50": round(float(p50), 3),
    "inference_ms_p95": round(float(p95), 3),
    "latency_threshold_ms": THRESHOLD_MS,
    "meets_latency_threshold": bool(p95 <= THRESHOLD_MS),
    "measured_rows": len(lat),
    "measure_note": "單筆 predict_proba，沙箱容器內、單執行緒",
}
Path("output/deployment_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
report

## 量測條件

在平台沙箱容器內、單執行緒，對驗證集 625 筆逐筆呼叫 predict_proba，量 wall-clock。p50 2.37ms、p95 7.82ms，門檻 200ms，通過。第一次呼叫有暖機成本，p95 已經包含。

## 精度與延遲的取捨

這個模型 val AUC 0.783、p95 7.8ms，離門檻還有 25 倍的空間，所以沒有犧牲精度去換速度。如果改用梯度提升樹，AUC 大約只差 0.01，延遲會增加數倍，沒有必要。

## 替代方案

若之後加入更多特徵、改成樹模型，p95 可能到 30–50ms，仍在門檻內；若要批次評分（每晚跑全部客戶），改成一次 predict_proba 整批，單筆成本會再降一個數量級。